# 00 · Overview & setup — CFTR Variant Toolkit

A beginner-friendly, **honest** tour of the computational tools used to interpret
CFTR variants. Each notebook takes one tool (or tool family), explains *what it
is*, *what its score means*, *the decision threshold and why*, and *how to get the
real data* — then notebook 12 combines them into the A1/A2 worklists and notebook
08 covers the methodology pitfalls.

This notebook just checks your setup and shows the honest data-provenance map.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))
import toolkit as tk
import pandas as pd, numpy as np
print("Python :", sys.version.split()[0])
print("toolkit:", pathlib.Path(tk.__file__).name)
print("cache  :", tk.CACHE_DIR, "->", "FOUND" if tk.CACHE_DIR.exists() else "MISSING")

Python : 3.13.14
toolkit: toolkit.py
cache  : <project>\_tmp_fetch -> FOUND


## The one thing to understand before anything else: REAL vs DEMO

This toolkit is deliberately transparent about what is a *real* download/prediction
and what is a small *hand-curated demo table* shipped so the notebooks run
anywhere. **Never quote a DEMO number as a finding.**

In [2]:
prov = pd.DataFrame([
  ("gnomAD v4 (allele freq)", "REAL",  "~2,466 missense / ~1,085 non-coding", "01"),
  ("AlphaMissense",           "REAL",  "genome-wide (all CFTR missense)",     "02"),
  ("EVE",                     "DEMO",  "~13 curated variants",                "03"),
  ("ESM1b",                   "DEMO",  "~13 curated variants",                "04"),
  ("REVEL",                   "DEMO",  "~13 curated variants",                "05"),
  ("PrimateAI",               "DEMO",  "~13 curated variants",                "06"),
  ("ClinVar",                 "REAL",  "genome-wide",                         "07"),
  ("CFTR2 (30 Jan 2026)",     "REAL",  "~2,097 variants / 780 missense keys", "08"),
  ("SpliceAI",                "DEMO",  "9 curated splice variants",           "09"),
  ("Pangolin",                "DEMO",  "9 curated splice variants",           "10"),
  ("CADD",                    "REAL",  "live per-variant API",                "11"),
], columns=["source","status","coverage","notebook"])
prov

,source,status,coverage,notebook
0,gnomAD v4 (allele freq),REAL,"~2,466 missense / ~1,085 non-coding",01
1,AlphaMissense,REAL,genome-wide (all CFTR missense),02
2,EVE,DEMO,~13 curated variants,03
3,ESM1b,DEMO,~13 curated variants,04
4,REVEL,DEMO,~13 curated variants,05
5,PrimateAI,DEMO,~13 curated variants,06
6,ClinVar,REAL,genome-wide,07
7,CFTR2 (30 Jan 2026),REAL,"~2,097 variants / 780 missense keys",08
8,SpliceAI,DEMO,9 curated splice variants,09
9,Pangolin,DEMO,9 curated splice variants,10


In [3]:
# quick proof the REAL loaders work against the cache
print("gnomAD missense :", len(tk.load_gnomad_missense()), "(REAL)")
print("AlphaMissense   :", len(tk.load_alphamissense()), "(REAL, genome-wide)")
print("ClinVar         :", len(tk.load_clinvar()), "(REAL)")
print("CFTR2           :", len(tk.load_cftr2()), "(REAL, 30 Jan 2026 release)")
print("EVE demo        :", len(tk.load_eve()), "(DEMO)")
print("splice demo     :", len(tk.load_splice_demo()), "(DEMO)")

gnomAD missense : 2466 (REAL)
AlphaMissense   : 8597 (REAL, genome-wide)
ClinVar         : 2801 (REAL)
CFTR2           : 2097 (REAL, 30 Jan 2026 release)
EVE demo        : 13 (DEMO)
splice demo     : 9 (DEMO)


## How to read this toolkit

One tool per notebook (03-11); the integration notebook (12) is where they are combined and compared.

| Notebook | Tool | Real? |
|---|---|---|
| **01** | gnomAD — population allele frequency | REAL |
| **02** | AlphaMissense — the one real genome-wide predictor | REAL |
| **03** | EVE — unsupervised evolutionary model | demo |
| **04** | ESM1b — protein language model (backwards scale) | demo |
| **05** | REVEL — supervised ensemble (+ circularity) | demo |
| **06** | PrimateAI — semi-supervised | demo |
| **07** | ClinVar — crowd-sourced clinical truth set | REAL |
| **08** | CFTR2 — functional truth set | **REAL** |
| **09** | SpliceAI — splice deltas | demo |
| **10** | Pangolin — independent splice model | demo |
| **11** | CADD — real, live deleteriousness score | REAL (live) |
| **12** | **Integration** — reproduce A1/A2 + cross-tool comparisons | mixed |
| **13** | **Circular reasoning & training leakage** — the methodology | — |

Recommended order: 01 → 13 in sequence. If you only read two: **12** (what the
numbers really are, plus the tool-vs-tool comparisons) and **13** (why to be careful).

## To swap in REAL data for the demo tools

Each tool notebook has a *"How to get the REAL data"* section. In short:

- **EVE** (03) — per-protein CSV from evemodel.org (CFTR = UniProt P13569)
- **ESM1b** (04) — bulk variant-effect files (Brandes 2023 supplement / HuggingFace)
- **REVEL** (05) — genome-wide table from sites.google.com/site/revelgenomics
- **PrimateAI** (06) — Illumina distribution
- **SpliceAI** (09) — precomputed VCF (Illumina BaseSpace) or the Broad lookup app
- **Pangolin** (10) — run locally (needs GPU)

Already REAL, no swap needed: gnomAD (01), AlphaMissense (02), ClinVar (07),
**CFTR2** (08, `data/cftr2_2026-01-30.csv`), CADD (11, live API).

Join each on `protein_variant` (missense) or genomic `chr/pos/ref/alt` (splice),
set the loader's `source` to `"REAL"`, and re-run notebook 12.